# Lesson 2 — Building Your First Agent with LangGraph

In the previous lesson, we learned how to build **chains**:
clear, composable pipelines where data flows from input to output.

Chains are powerful, but they follow a **straight line**.
Each step runs once, in a fixed order, and then stops.

An **agent** works differently.

An agent does not just execute instructions.
It operates in a **loop**.

It:
- receives information,
- reasons about what to do next,
- takes an action,
- observes the result of that action,
- and uses this new information to decide again.

This cycle is often described as:

> **think → act → observe → think again**

This looping behavior is what gives agents their sense of autonomy.


In [2]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, List
import operator

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

from langgraph.graph import StateGraph, END

load_dotenv()
GPT_MODEL = "gpt-5-nano"
ANTHROPIC_MODEL = "claude-opus-4-6"

/data/venvs/py312-dl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Why LangGraph ? Moving from Chains to Stateful Graphs

LCEL chains are powerful, but by design they form a **Directed Acyclic Graph (DAG)**.

In practice, this means:
- data flows forward,
- each step runs once,
- execution always moves toward a fixed end.

This structure is ideal for **pipelines**:
for example, formatting input → calling a model → parsing output.

However, this is not how real-world systems behave when they must **adapt**.

Consider a real application:

- A user asks for travel recommendations.
- The system queries a flight API.
- The API returns incomplete results.
- The system needs to call a hotel API.
- If prices are too high, it tries a different date.
- If everything looks good, it responds to the user.

This process is **not linear**.
It involves:
- decisions,
- retries,
- conditional paths,
- and memory of previous attempts.

A simple input-to-output chain cannot express this behavior.
Once a chain finishes, it stops.

> **The Agent Loop: Think, Act, Observe, Loop**

Agent behavior is best described as a loop with four phases.
Each phase maps directly to real-world system behavior.

### 1. Think

The agent analyzes the current situation.

In real applications, this can mean:
- reading the user request,
- inspecting stored context or memory,
- checking which data is already available.

Example:
> A backend service checks whether it already has user preferences  
> before deciding to call an external recommendation API.

This step is about **decision-making**, not action.

### 2. Act

Based on its reasoning, the agent performs an action.

In practice, this often corresponds to:
- calling an external API,
- querying a database,
- sending a request to another service,
- triggering a background job.

Example:
> The agent calls a weather API to verify conditions  
> before suggesting outdoor activities.

This step produces **new information**.

### 3. Observe

The agent receives the result of its action.

This result may be:
- an API response,
- a database result,
- an error or timeout,
- partial or unexpected data.

Example:
> The weather API responds with missing data for a location,  
> or returns an error due to rate limits.

The agent stores this outcome in its state.

### 4. Loop

Using the updated state, the agent decides what to do next.

Possible decisions include:
- retrying with different parameters,
- calling a fallback service,
- asking the user for clarification,
- stopping and returning a final answer.

Example:
> If the weather data is incomplete,  
> the agent calls a second API or switches strategy.

This loop may run several times until the goal is reached.

> **Why LangGraph Is Needed ?**

This kind of behavior cannot be modeled with a DAG.

Agents require:
- persistent state,
- conditional transitions,
- explicit loops,
- and clear control over execution flow.

LangGraph is designed specifically for this purpose.

> **How LangGraph Models Real Agent Behavior ?**

LangGraph represents agent logic as a **stateful graph**, built from:

- **Nodes**  
  Individual steps such as reasoning, acting, or processing results.  
  Example: calling an API, analyzing a response, updating memory.

- **Edges**  
  Transitions that define what happens next.  
  Example: continue, retry, switch strategy, or stop.

- **State**  
  Shared memory that persists across steps.  
  Example: API responses, counters, errors, user preferences.

This structure mirrors how real backend workflows and autonomous systems operate.

To make these ideas concrete, we will now build a **minimal LangGraph example**.

The goal of the example is not to solve a real business problem yet,
but to clearly show:
- how state is updated,
- how decisions are made,
- and how a loop is expressed explicitly in code.

Once this structure is clear,
adding LLM reasoning and real tools will feel natural.

### Defining the State

The state represents the memory of the graph.
It is passed to every node and updated over time.

Here, the state contains a single value: a counter.

In [ ]:
class MyGraphState(TypedDict):
    """
    Shared state of the graph.
    """
    counter: Annotated[int, operator.add]

The `Annotated[int, operator.add]` tells LangGraph
how to merge updates coming from different steps.


### Defining the Nodes

Nodes are plain Python functions.
Each node:
- receives the current state,
- performs some logic,
- returns a partial update to the state.

In a real agent:
- one node might call an API,
- another might analyze a response,
- another might decide the next action.

In [23]:
def start_node(state: MyGraphState):
    """Entry point of the graph."""
    print("--- START NODE ---")
    return {"counter": 1}

def count_node(state: MyGraphState):
    """Increment the counter."""
    print("--- COUNT NODE ---")
    current_count = state["counter"]
    print(f"Current count: {current_count}")
    return {"counter": 1}

### Adding Decision Logic with a Conditional Edge

Agents must decide whether to continue or stop.

In real applications, this decision could depend on:
- an API response status,
- missing or complete data,
- an error threshold,
- a user-defined limit.

Here, we simulate this logic with a simple counter.

In [29]:
def should_continue(state: MyGraphState):
    """Decide whether the graph should continue or end."""
    print("--- CONDITIONAL EDGE ---")
    counter = state["counter"]
    print(f"Checking counter: {counter}")

    if counter < 5:
        print("Decision: continue")
        return "continue"
    else:
        print("Decision: end")
        return "end"

### Building the Graph Structure

With the state and nodes defined,
we can now assemble the graph.

This step defines:
- which node runs first,
- how nodes are connected,
- where the loop happens,
- and how execution ends.

In [30]:
workflow = StateGraph(state_schema=MyGraphState)

# Register nodes
workflow.add_node("start", start_node)
workflow.add_node("count_up", count_node)

# Define the entry point
workflow.set_entry_point("start")

# Connect nodes
workflow.add_edge("start", "count_up")

# Add a conditional loop
workflow.add_conditional_edges(
    source="count_up",
    path=should_continue,
    path_map={
        "continue": "count_up",
        "end": END,
    },
)

### Compiling and Running the Graph

Once the structure is defined,
the graph is compiled into an executable application.

Running the graph triggers the full loop:
state updates, decisions, and termination.


In [31]:
app = workflow.compile()

print("Running the graph...")
final_state = app.invoke({})

print("\n--- FINAL STATE ---")
print(final_state)

Running the graph...
--- START NODE ---
--- COUNT NODE ---
Current count: 1
--- CONDITIONAL EDGE ---
Checking counter: 2
Decision: continue
--- COUNT NODE ---
Current count: 2
--- CONDITIONAL EDGE ---
Checking counter: 3
Decision: continue
--- COUNT NODE ---
Current count: 3
--- CONDITIONAL EDGE ---
Checking counter: 4
Decision: continue
--- COUNT NODE ---
Current count: 4
--- CONDITIONAL EDGE ---
Checking counter: 5
Decision: end

--- FINAL STATE ---
{'counter': 5}


Even without an LLM, this graph already behaves like an agent:

- it keeps memory across steps,
- it decides whether to continue or stop,
- it repeats actions based on state,
- it exits when a condition is met.

If you replace:
- `count_node` with an API call,
- `should_continue` with a validation check,

you already have the core of a real agent workflow.


From here, we are ready to build agents that reason, act, and adapt.

## Core Concepts: State, Nodes, and Edges

To build powerful and reliable agents with LangGraph, it is essential to clearly understand its three architectural pillars:

- **State**
- **Nodes**
- **Edges**

These three elements correspond directly to how real systems behave.

In a production environment:
- the **state** stores memory (API responses, conversation history, counters),
- **nodes** execute logic (LLM calls, API calls, validations),
- **edges** define the control flow (continue, retry, stop, branch).

Let’s examine each one in detail.

### State: The Agent's Memory

The state represents everything the agent knows at a given moment.

In real-world systems, this could include:
- the user's request,
- conversation history,
- API responses,
- retry counters,
- tool outputs,
- intermediate reasoning steps.

In LangGraph, the state is defined using a `TypedDict`.
This schema tells the graph what fields exist and how they should be merged.

In [2]:
class AgentState(TypedDict):
    """
    A state object for a simple conversational agent.
    """
    input: str
    messages: Annotated[List[str], operator.add]
    next_action: str

Two important things happen here:

1. `messages` uses `Annotated[List[str], operator.add]`.  
   This means every time a node returns `{"messages": [...]}`,  
   the new list is appended to the existing one.

2. `next_action` stores the decision that will determine which edge is taken next.

This pattern is common in real agents:
- accumulate conversation history,
- compute the next action,
- transition accordingly.

### Nodes: The Agent's Actions

Nodes are functions that perform work.

They:
- receive the current state,
- execute some logic,
- return partial updates to the state.

In production systems, a node might:
- call an LLM,
- query a database,
- call a weather API,
- validate data,
- or update memory.

In [3]:
def call_language_model(state: AgentState):
    """
    A node that simulates calling a language model.
    """
    print("--- NODE: CALLING LANGUAGE MODEL ---")
    last_message = state["messages"][-1]
    
    if "hello" in last_message.lower():
        response = "Hello! How can I help you today?"
        next_action = "get_user_response"
    else:
        response = "I'm sorry, I don't understand."
        next_action = "end"
    
    return {
        "messages": [response],
        "next_action": next_action,
    }

This node demonstrates a common agent pattern:

- Inspect the latest message.
- Generate a response.
- Decide what should happen next.

The returned dictionary updates:
- `messages` (which accumulates history),
- `next_action` (which will guide the next edge).

This mirrors real LLM-based decision steps.

### Edges: The Agent's Logic

Edges define how execution moves between nodes.

LangGraph provides two main mechanisms:

1. **Standard edges** → `workflow.add_edge()`
2. **Conditional edges** → `workflow.add_conditional_edges()`

Standard edges are simple transitions:
always go from A to B.

Conditional edges introduce decision logic.
They allow branching based on state.

### Conditional Edges in Detail

A conditional edge requires three elements:

- **A source node** : 
  Defined when calling `workflow.add_conditional_edges(source=...)`

- **A decision function** :
  A function that reads the state and returns a label. This label is the "decision"

- **A mapping dictionary** : A dictionary that maps each possible decision label to a decision node.

## Putting Everything Together

We now build a small but complete agent.

This example simulates:
- asking for a name,
- greeting the user,
- repeating the greeting multiple times,
- stopping after a defined number of iterations.

This could represent a real scenario such as:
- retrying an API call,
- polling a service,
- sending follow-up notifications.

In [40]:
# 1. Define the State

class GreeterState(TypedDict):
    name: str
    greeting_count: Annotated[int, operator.add]
    conversation: Annotated[List[str], operator.add]

- `greeting_count` accumulates how many greetings were sent.
- `conversation` accumulates all messages.

In [41]:
# 2. Define the Nodes

def get_name(state: GreeterState):
    print("--- NODE: GET NAME ---")
    
    return {
        "name": "Yatoute",
        "conversation": ["System: What's your name?"]
    }

def greet_user(state: GreeterState):
    print("--- NODE: GREET USER ---")
    
    name = state["name"]
    greeting_count = state["greeting_count"]
    
    greeting = f"Hello, {name}! This is my {greeting_count + 1} greeting."
    
    print(f"Generated greeting: '{greeting}'")
    
    return {
        "conversation": [greeting],
        "greeting_count": 1,
    }

Notice how:

- `greeting_count` increments automatically because of `operator.add`.
- Each new greeting is appended to `conversation`.

In [43]:
# 3. Define the Conditional Edge

def should_greet_again(state: GreeterState):
    print("--- CONDITIONAL EDGE: SHOULD GREET AGAIN ---")
    
    greeting_count = state["greeting_count"]
    
    print(f"Checking greeting count: {greeting_count}")
    
    if greeting_count < 3:
        print("Decision: continue greeting")
        return "continue_greeting"
    else:
        print("Decision: stop greeting")
        return "stop_greeting"

In [7]:
# 4. Build the Graph

workflow = StateGraph(state_schema=GreeterState)

workflow.add_node("fetch_name", get_name)
workflow.add_node("say_hello", greet_user)

workflow.set_entry_point("fetch_name")

workflow.add_edge("fetch_name", "say_hello")

workflow.add_conditional_edges(
    source="say_hello",
    path=should_greet_again,
    path_map={
        "continue_greeting": "say_hello",
        "stop_greeting": END,
    }
)

In [8]:
# 5. Compile and Run

app = workflow.compile()

print("Running the graph...")
initial_state = {"greeting_count": 0}

final_state = app.invoke(initial_state)

print("\n--- FINAL STATE ---")
print(final_state)

Running the graph...
--- NODE: GET NAME ---
--- NODE: GREET USER ---
Generated greeting: 'Hello, Yatoute! This is my 1 greeting.'
--- CONDITIONAL EDGE: SHOULD GREET AGAIN ---
Checking greeting count: 1
Decision: continue greeting
--- NODE: GREET USER ---
Generated greeting: 'Hello, Yatoute! This is my 2 greeting.'
--- CONDITIONAL EDGE: SHOULD GREET AGAIN ---
Checking greeting count: 2
Decision: continue greeting
--- NODE: GREET USER ---
Generated greeting: 'Hello, Yatoute! This is my 3 greeting.'
--- CONDITIONAL EDGE: SHOULD GREET AGAIN ---
Checking greeting count: 3
Decision: stop greeting

--- FINAL STATE ---
{'name': 'Yatoute', 'greeting_count': 3, 'conversation': ["System: What's your name?", 'Hello, Yatoute! This is my 1 greeting.', 'Hello, Yatoute! This is my 2 greeting.', 'Hello, Yatoute! This is my 3 greeting.']}


This small graph already behaves like an agent:

- It stores memory in the state.
- It updates memory across iterations.
- It makes decisions.
- It loops based on conditions.
- It stops when a rule is satisfied.

If you replace:
- `greet_user` with an LLM call,
- `should_greet_again` with a validation of API results,

you now have a realistic backend agent pattern.

This structure scales naturally to:
- tool-based agents,
- retry logic,
- planning agents,
- multi-step workflows.

Understanding these three core concepts
(State, Nodes, and Edges)
is the foundation of all LangGraph agents.

## Building a Basic Agentic Loop: The "Hello, Agent!" Project

We now have the building blocks:
- **State** (memory),
- **Nodes** (actions),
- **Edges** (control flow).

It is time to combine them into a first real agent loop.

This first project will stay intentionally simple:
- no external tools yet,
- no web search,
- no database calls.

The agent will behave like a basic conversationalist.

Even so, building it with LangGraph matters.
It creates the foundation that later supports:
- tool calling,
- multi-step reasoning,
- retries and fallbacks,
- parallel workflows.

### The Workflow for One Turn

For a single “turn” of interaction, the loop is straightforward:

1. The user provides an initial message.
2. The message becomes the initial **State**.
3. The entry point sends execution to a node that calls an LLM.
4. The LLM generates a response.
5. The node updates the state by appending the response to the conversation history.
6. The graph ends for this turn.

This is the smallest useful agent loop:  
**input → think → update memory → return**.

### 1) Define the Agent State

The agent state stores the conversation.

We use `Annotated[..., operator.add]` so new messages returned by nodes
are **appended** to the existing list.

This mirrors real chat agents:
the full conversation history stays available at every turn.

In [2]:
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]

### 2) Define the Model Node (OpenAI)

This node is the agent’s “thinking” step.
It sends the current conversation to an OpenAI chat model.

In real applications, this node could also:
- add system instructions,
- inject retrieved context,
- or decide which tools to call next.

For now, it only generates a response and returns it as a state update.

In [36]:
def call_openai(state: AgentState):
    """
    Calls an OpenAI model using the conversation stored in state.
    """
    print("--- NODE: call_openai (OpenAI) ---")

    model = ChatOpenAI(model=GPT_MODEL, temperature=0)
    response = model.invoke(state["messages"])

    print(f"Generated message (OpenAI): {response.content}")
    return {"messages": [response]}

### 3) Add a Second Model Node (Claude)

To make this project more realistic, we add another LLM node.

This reflects a common production pattern:
- run a cheaper model first,
- use a stronger model for validation,
- or compare outputs for safety and quality.

Here, we simply demonstrate how to plug in a second provider.

In [26]:
def call_claude(state: AgentState):
    """
    Calls an Anthropic model (Claude) using the conversation stored in state.
    """
    print("--- NODE: call_claude (Claude) ---")

    model = ChatAnthropic(model=ANTHROPIC_MODEL, temperature=0)
    response = model.invoke(state["messages"])

    print(f"Generated message (Claude): {response.content}")
    return {"messages": [response]}

### 4) Wiring the Graph

For this first loop, we will support two modes:
- run OpenAI and stop,
- run Claude and stop.

Later in the course, we will add real decision logic
that chooses which model to call based on the state.

In [37]:
workflow = StateGraph(AgentState)

workflow.add_node("openai", call_openai)
workflow.add_node("claude", call_claude)

# We choose OpenAI as the default entry point for now
workflow.set_entry_point("openai")

# End after one model call (one-turn loop)
workflow.add_edge("openai", END)
workflow.add_edge("claude", END)

In [38]:
app = workflow.compile()

### 5) Running the Agent (OpenAI)

We start with a single user message stored as a `HumanMessage`.
This becomes the initial memory of the agent.

In [21]:
initial_input = {
    "messages": [
        HumanMessage(content="Hello! Can you help me make a coffee ? Be short !")
    ]
}

print("--- Invoking Agent (OpenAI) ---")
final_state = app.invoke(initial_input)

print("\n--- Final State (last message) ---")
print(final_state["messages"][-1].content)

--- Invoking Agent (OpenAI) ---
--- NODE: call_openai (OpenAI) ---
Generated message (OpenAI): Sure! Quick options—tell me which you have:

- Instant: 1–2 tsp coffee + 180 ml hot water, stir.
- Drip/Pourover: ~7–8 g coffee, 180 ml water (92–96°C); bloom 30 s, pour to finish (4 min total).
- French press: ~17 g coffee + 250 ml water; 4 min steep, press, pour.
- Espresso: ~18–20 g for a double shot, 36–40 ml, 25–30 s extraction.

What equipment do you have?

--- Final State (last message) ---
Sure! Quick options—tell me which you have:

- Instant: 1–2 tsp coffee + 180 ml hot water, stir.
- Drip/Pourover: ~7–8 g coffee, 180 ml water (92–96°C); bloom 30 s, pour to finish (4 min total).
- French press: ~17 g coffee + 250 ml water; 4 min steep, press, pour.
- Espresso: ~18–20 g for a double shot, 36–40 ml, 25–30 s extraction.

What equipment do you have?


### Switching to Claude

To run the Claude node instead, set the entry point to `"claude"`.

This is a simple change, but it shows an important design benefit:
the state and graph structure remain the same,
only the node implementation changes.

In [22]:
workflow_claude = StateGraph(AgentState)
workflow_claude.add_node("openai", call_openai)
workflow_claude.add_node("claude", call_claude)

workflow_claude.set_entry_point("claude")
workflow_claude.add_edge("openai", END)
workflow_claude.add_edge("claude", END)

app_claude = workflow_claude.compile()

print("--- Invoking Agent (Claude) ---")
final_state_claude = app_claude.invoke(initial_input)

print("\n--- Final State (last message) ---")
print(final_state_claude["messages"][-1].content)

--- Invoking Agent (Claude) ---
--- NODE: call_claude (Claude) ---
Generated message (Claude): Sure! Here's a basic **drip coffee** method:

1. **Boil** fresh water
2. **Add** 2 tbsp ground coffee per cup into filter
3. **Pour** hot water over grounds
4. **Wait** ~4 min
5. **Enjoy!**

Want a specific method (French press, espresso, etc.)?

--- Final State (last message) ---
Sure! Here's a basic **drip coffee** method:

1. **Boil** fresh water
2. **Add** 2 tbsp ground coffee per cup into filter
3. **Pour** hot water over grounds
4. **Wait** ~4 min
5. **Enjoy!**

Want a specific method (French press, espresso, etc.)?


### Analyzing the "Hello, Agent!" Loop

Let’s break down what happened when `app.invoke()` was called.
This analysis is exactly how you debug real agents later.

1. **State Initialization**  
   The process started with `app.invoke(initial_input)`.  
   The state contained a list of messages with one element: a `HumanMessage`.  
   This is the agent’s initial memory.

2. **Entry Point Selection**  
   The entry point (`workflow.set_entry_point(...)`) decides which node runs first.  
   In the first run, the graph started at `"openai"`.

3. **Node Execution**  
   The node function received the full state:  
   `state["messages"]` was passed into the LLM call (`model.invoke(...)`).

4. **Model Response**  
   The model returned a new `AIMessage` response object.

5. **State Update**  
   The node returned `{"messages": [response]}`.  
   Because `messages` uses `operator.add`, LangGraph appended the response  
   to the existing conversation list.

6. **Edge to END**  
   The graph followed `workflow.add_edge(<node>, END)`  
   and stopped execution for this turn.

At this point, we have a working agent loop:
**memory → model call → memory update → stop**.

> **What We Achieved ?**

Even without tools, this is already an agentic structure:
- a persistent state schema,
- a reasoning/action node,
- explicit control flow.

Adding a second model node (Claude) makes the structure closer to real systems:
you can now compare models, add fallback logic, or route requests.

## Compiling and Running Your Graph

Defining a graph is only half the job.

To actually execute it, LangGraph uses two key steps:

- **Compile** the graph structure into a runnable app: `workflow.compile()`
- **Run** the app using one of the execution methods (`invoke`, `stream`, async variants)

Compilation is the moment when LangGraph:
- checks your state schema,
- prepares the execution plan,
- and registers how state updates should be merged.

After compilation, the graph becomes an executable object (often called `app`).

To make execution modes easy to understand, we will reuse the looping **Greeter** graph.

Even though it is a toy example, the structure is realistic:
it behaves like a backend workflow that repeats actions until a stopping condition is met.

### Running the Graph with `.invoke()`

The most direct way to run a compiled graph is `.invoke()`.

- execution starts at the entry point (`workflow.set_entry_point(...)`)
- nodes run in order, following edges
- the process stops when the graph reaches `END`
- the return value is the **final state**

The input to `.invoke()` must be a dictionary that matches your state schema.
In practice, this is what your agent “knows” at the start of the run.

In [44]:
workflow = StateGraph(GreeterState)

workflow.add_node("fetch_name", get_name)
workflow.add_node("say_hello", greet_user)

workflow.set_entry_point("fetch_name")

workflow.add_edge("fetch_name", "say_hello")

workflow.add_conditional_edges(
    source="say_hello",
    path=should_greet_again,
    path_map={
        "continue_greeting": "say_hello",
        "stop_greeting": END,
    },
)

app = workflow.compile()

Now we provide the initial state and run the full graph.

This mirrors a real backend call:
the system starts with an initial context (state),
then runs the full workflow until completion.

In [45]:
initial_state = {"greeting_count": 0}
final_state = app.invoke(initial_state)

print("\n--- FINAL STATE ---")
print(final_state)

--- NODE: GET NAME ---
--- NODE: GREET USER ---
Generated greeting: 'Hello, Yatoute! This is my 1 greeting.'
--- CONDITIONAL EDGE: SHOULD GREET AGAIN ---
Checking greeting count: 1
Decision: continue greeting
--- NODE: GREET USER ---
Generated greeting: 'Hello, Yatoute! This is my 2 greeting.'
--- CONDITIONAL EDGE: SHOULD GREET AGAIN ---
Checking greeting count: 2
Decision: continue greeting
--- NODE: GREET USER ---
Generated greeting: 'Hello, Yatoute! This is my 3 greeting.'
--- CONDITIONAL EDGE: SHOULD GREET AGAIN ---
Checking greeting count: 3
Decision: stop greeting

--- FINAL STATE ---
{'name': 'Yatoute', 'greeting_count': 3, 'conversation': ["System: What's your name?", 'Hello, Yatoute! This is my 1 greeting.', 'Hello, Yatoute! This is my 2 greeting.', 'Hello, Yatoute! This is my 3 greeting.']}


When you run this code, `.invoke()` executes the entire loop.
The graph keeps looping until `should_greet_again` returns `"stop_greeting"`,
which routes execution to `END`.

This is similar to real workflows such as:
- retrying an API call up to N times,
- polling a service until a job is finished,
- iterating through items until a condition is met.

### Streaming Intermediate Steps with `.stream()`

In many situations, the final result is not enough.

When you are:
- debugging an agent,
- building a UI (show progress),
- or teaching how the loop works,

you often want to see what happens **after each node**.

This is exactly what `.stream()` provides.

Instead of returning only the final state,
`.stream()` yields intermediate results step by step.
Each yielded item contains:
- which node just finished,
- and what the state looked like after that node.

In [47]:
initial_state = {"greeting_count": 0}

for step in app.stream(initial_state):

    node_name = list(step.keys())[0]
    state_after_node = step[node_name]

    print(f"--- Node '{node_name}' finished ---")
    print("--- State after this step ---")
    print(state_after_node)

print("\n--- STREAMING COMPLETED ---")

--- NODE: GET NAME ---
--- Node 'fetch_name' finished ---
--- State after this step ---
{'name': 'Yatoute', 'conversation': ["System: What's your name?"]}
--- NODE: GREET USER ---
Generated greeting: 'Hello, Yatoute! This is my 1 greeting.'
--- CONDITIONAL EDGE: SHOULD GREET AGAIN ---
Checking greeting count: 1
Decision: continue greeting
--- Node 'say_hello' finished ---
--- State after this step ---
{'conversation': ['Hello, Yatoute! This is my 1 greeting.'], 'greeting_count': 1}
--- NODE: GREET USER ---
Generated greeting: 'Hello, Yatoute! This is my 2 greeting.'
--- CONDITIONAL EDGE: SHOULD GREET AGAIN ---
Checking greeting count: 2
Decision: continue greeting
--- Node 'say_hello' finished ---
--- State after this step ---
{'conversation': ['Hello, Yatoute! This is my 2 greeting.'], 'greeting_count': 1}
--- NODE: GREET USER ---
Generated greeting: 'Hello, Yatoute! This is my 3 greeting.'
--- CONDITIONAL EDGE: SHOULD GREET AGAIN ---
Checking greeting count: 3
Decision: stop greeting

A real-world analogy:

- `.invoke()` is like calling an API endpoint and only getting the final JSON response.
- `.stream()` is like getting live server events (progress updates) while the workflow runs.

For agentic systems, streaming is especially useful because:
- you can display the agent’s progress,
- you can debug looping behavior,
- you can inspect state evolution at every step.

### Key Idea

Both `.invoke()` and `.stream()` run the same graph.

The difference is visibility:

- `.invoke()` gives the final state.
- `.stream()` gives intermediate states after each node.

This distinction becomes critical once your agents:
- call multiple tools,
- run several loops,
- or take minutes to complete a task.

## Visualizing and Tracing the Agent’s Path with LangSmith

As agents grow more complex, a common problem appears:

> “What did the agent actually do, and why?”

With loops, conditional edges, and multiple model/tool calls, prints in the notebook become hard to follow.
This is where **LangSmith tracing** becomes useful.

A trace gives you a structured view of execution:
- which nodes ran (and in what order),
- which edges were taken,
- what inputs/outputs were produced at each step,
- how the state changed over time.

In real projects, this is essential for:
- debugging unexpected loops,
- understanding failures (timeouts, bad tool outputs),
- evaluating model behavior across many runs.

### Setting Up Automatic Tracing (Environment Variables)

LangGraph integrates with LangSmith almost automatically.

Once the right environment variables are set, every run of a compiled graph can be logged
without changing your graph logic.

Add these variables to your `.env` file:

In [ ]:
# .env (example)
LANGSMITH_TRACING="true"
LANGSMITH_ENDPOINT="https://eu.api.smith.langchain.com"
LANGSMITH_PROJECT="default"
LANGSMITH_API_KEY="YOUR_LANGSMITH_API_KEY"

What each variable does:

- `LANGSMITH_TRACING="true"`  
  Enables tracing. Without this, nothing is logged.

- `LANGSMITH_ENDPOINT="https://eu.api.smith.langchain.com"`  
  Tells LangSmith where to send traces (EU endpoint here).

- `LANGSMITH_PROJECT="default"`  
  Traces will appear under this project name in the UI.

- `LANGSMITH_API_KEY="..."`  
  Authenticates your requests to LangSmith.

With these values set, tracing becomes “always on” for your runs.


### Running a Graph with Tracing Enabled

We now run a minimal one-turn agent (Claude node).
The code is the same as before: the only difference is that tracing is enabled via environment variables.

Behind the scenes, LangGraph will report execution details to LangSmith:
- the node call,
- the model invocation,
- the state update,
- the transition to `END`.


In [5]:
load_dotenv()

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]

def call_claude(state: AgentState):
    """
    Calls an Anthropic model (Claude) using the conversation stored in state.
    """
    print("--- NODE: call_claude (Claude) ---")

    model = ChatAnthropic(model=ANTHROPIC_MODEL, temperature=0)
    response = model.invoke(state["messages"])

    print(f"Generated message (Claude): {response.content}")
    return {"messages": [response]}

workflow_claude = StateGraph(AgentState)
workflow_claude.add_node("claude", call_claude)

workflow_claude.set_entry_point("claude")
workflow_claude.add_edge("claude", END)

app_claude = workflow_claude.compile()

initial_input = {
    "messages": [
        HumanMessage(content="Hello! Can you help me make a coffee ? Be short !")
    ]
}

print("--- Invoking Agent (Claude) ---")
final_state_claude = app_claude.invoke(initial_input)

print("\n--- Final State (last message) ---")
print(final_state_claude["messages"][-1].content)

--- Invoking Agent (Claude) ---
--- NODE: call_claude (Claude) ---
Generated message (Claude): Sure! Here's a basic **drip coffee** method:

1. **Boil** fresh water
2. **Add** 2 tbsp ground coffee per cup into filter
3. **Pour** hot water over grounds
4. **Wait** ~4 min
5. **Enjoy!**

Want a specific method (French press, espresso, etc.)?

--- Final State (last message) ---
Sure! Here's a basic **drip coffee** method:

1. **Boil** fresh water
2. **Add** 2 tbsp ground coffee per cup into filter
3. **Pour** hot water over grounds
4. **Wait** ~4 min
5. **Enjoy!**

Want a specific method (French press, espresso, etc.)?


### Exploring the Trace in the LangSmith UI

After running the code, open your LangSmith project in the web UI (https://eu.smith.langchain.com/).
You should see a new run corresponding to this agent execution.

In the trace view, two panels are especially useful:

1) **Component Tree (left panel)**  
   This shows the execution structure, typically like:
   - `StateGraph` (top-level run)
   - the node call (e.g., `call_claude`)
   - the model call (the LLM invocation)

   This is useful when the agent has loops:
   you can immediately see repeated node executions.

2) **Details View (right panel)**  
   This shows inputs and outputs at each level:
   - the state passed into the node,
   - the model prompt/messages,
   - the model response,
   - the state update returned by the node.

This is extremely helpful when debugging real scenarios such as:
- “Why did the agent loop 10 times?”
- “Which node produced the wrong state value?”
- “Which model/tool call failed, and with what error?”


## Conclusion — From Structured Graphs to Real-World Agents

At this point, you can:

- define a clear **state schema** for your agent,
- implement **nodes** that perform reasoning or actions,
- connect them using **edges** and conditional logic,
- run full execution loops with `.invoke()` or inspect them step by step with `.stream()`,
- and trace the entire execution path using LangSmith.

More importantly, you now understand a key idea:

An agent is not just a model call.  
It is a **stateful control system** that decides what happens next.

With LangGraph, you have learned how to:

- represent memory explicitly,
- control execution flow,
- model loops and decisions,
- structure agent behavior in a transparent and debuggable way.

This is the architectural foundation of real-world AI systems.

So far, our agents have been conversational.
They can reason, respond, and update memory — but they do not yet interact with the outside world.

In the next lesson, we will extend this structure by introducing **tools**.

We will explore how to:

- connect agents to external APIs,
- query databases or search engines,
- control when and how tools are called,
- integrate tool results back into the agent’s state.

This is the moment when agents move from *talking*  
to *acting*.

Lesson 2 gave you control flow.  
The next lesson will give your agent real-world capabilities.
